### Library imports

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib
import os

### Training Model

In [7]:

print("Loading dataset...")
df = pd.read_csv("../data/SMSSpamCollection.csv", encoding="latin-1")

df = df[['v1', 'v2']]
df.columns = ['label', 'text']

print("Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], 
    df['label'], 
    test_size=0.2, 
    random_state=42  # This makes it reproducible
)

print("Vectorizing text (TF-IDF)...")
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

print("Training Naive Bayes classifier...")
model = MultinomialNB()
model.fit(X_train_vectorized, y_train)

print("Evaluating model...")
predictions = model.predict(X_test_vectorized)

# Calculate metrics
accuracy = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions, pos_label='spam')

print("-" * 30)
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1 * 100:.2f}%")
print("-" * 30)

print("Exporting Model...")
# Saves the trained vectorizer and model to disk
os.makedirs("models", exist_ok=True)
joblib.dump(vectorizer, "../artifacts/models/tfidf_vectorizer.joblib")
joblib.dump(model, "../artifacts/models/naive_bayes_model.joblib")
print("Models saved to the artifacts directory.")




Loading dataset...
Splitting data...
Vectorizing text (TF-IDF)...
Training Naive Bayes classifier...
Evaluating model...
------------------------------
Accuracy: 97.31%
F1 Score: 88.89%
------------------------------
Exporting Model...
Models saved to the artifacts directory.


### Reports

In [8]:
print("Classification Report:")
print(classification_report(y_test, predictions))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       965
        spam       1.00      0.80      0.89       150

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115

Confusion Matrix:
[[965   0]
 [ 30 120]]


In [9]:
import numpy as np
from sklearn.metrics import f1_score, precision_recall_curve

probabilities = model.predict_proba(X_test_vectorized)[:, 1]  # prob of spam

# Try a range of thresholds and find the one that maximizes F1
thresholds = np.arange(0.1, 0.5, 0.02)
best_f1, best_threshold = 0, 0.5

for t in thresholds:
    preds = (probabilities >= t).astype(int)
    preds_labels = np.where(preds == 1, 'spam', 'ham')
    f1 = f1_score(y_test, preds_labels, pos_label='spam')
    if f1 > best_f1:
        best_f1, best_threshold = f1, t

print(f"Best threshold: {best_threshold:.2f}, F1: {best_f1:.4f}")

Best threshold: 0.20, F1: 0.9392


In [ ]:
def classify_text(payload: PredictionRequest):
    ...
    text_matrix = vectorizer.transform([payload.text])
    probability_array = model.predict_proba(text_matrix)[0]
    spam_probability = probability_array[1]  # index 1 = spam (alphabetical order)

    predicted_label = "spam" if spam_probability >= threshold else "ham"
    confidence_score = float(spam_probability if predicted_label == "spam" else 1 - spam_probability)

    return PredictionResponse(
        prediction=predicted_label,
        confidence_score=round(confidence_score, 4)
    )